In [2]:
from jax import config
config.update("jax_enable_x64", True)
import jax
import jax.numpy as jnp
from torch.onnx.symbolic_opset9 import tensor

from tensordev.core import Jax, JaxShuffleCore

%load_ext autoreload
%autoreload 2

In [9]:
# --- SETUP ---
core = Jax()

# Base dimension and max INPUT degree of A / B
d = 2
input_deg = 2

# N = max OUTPUT degree = deg(A) + deg(B).
# precompute_shuffle must be called with N = input_deg + input_deg before any
# jax.jit trace so the full operator set is available at trace time.
shuffle_core = JaxShuffleCore(d, input_deg*2)

# Set up example tensors — degrees 0, 1, 2  (batch=2, widths 1, d, d²)
A = (jnp.array([[1.0],[1.0]]), jnp.array([[0.0,1.0],[1.0,0.0]]), jnp.array([[2.0,3.0,0.0,0.0],[0.0,0.0,0.0,0.0]]))
B = (jnp.array([[2.0],[2.0]]), jnp.array([[2.0,0.0],[3.0,0.0]]), jnp.array([[1.0,0.0,0.0,1.0],[0.0,0.0,0.0,0.0]]))


A = tuple(a[None, :, :] for a in A)
B = tuple(b[None, :, :] for b in B)

In [10]:
# --- COMPUTE SHUFFLE PRODUCT ---
# sa is passed as a JAX-static argument (identity-hashed ShuffleAlgebra);
# the result is JIT-compiled on the first call and cached for subsequent calls.
res = shuffle_core.tensor_shuffle_product(A, B)
res  # Correct shuffle product of A and B

(Array([[[2.],
         [2.]]], dtype=float64),
 Array([[[2., 2.],
         [5., 0.]]], dtype=float64),
 Array([[[5., 8., 2., 1.],
         [6., 0., 0., 0.]]], dtype=float64),
 Array([[[12., 13.,  7.,  0.,  1.,  0.,  0.,  3.],
         [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.]]], dtype=float64),
 Array([[[12.,  9.,  6.,  2.,  3.,  2.,  2.,  9.,  0.,  2.,  2.,  6.,
           2.,  3.,  0.,  0.],
         [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
           0.,  0.,  0.,  0.]]], dtype=float64))

In [12]:
# --- COMPUTE GRADIENT INVOLVING SHUFFLE PRODUCT ---
# Shows how to differentiate through the shuffle product end-to-end.
#
# sa is closed over (not a parameter of loss_fn) so jax.grad only
# differentiates w.r.t. A and B, not the static operators.

# f(A, B) = mean_over_batch( sum_k ||（A ⊔ B)_k||² )
def loss_fn(A: tuple, B: tuple):
    C = shuffle_core.tensor_shuffle_product(A, B)
    batch_total_sq = sum(jnp.sum(jnp.square(c_k), axis=-1) for c_k in C)
    return jnp.sum(batch_total_sq) / A[0].shape[0]

# Inspect value
print(f"{loss_fn(A, B) = }")

# Gradient w.r.t. both A and B
grad_fn = jax.grad(loss_fn, argnums=(0, 1))
grad_A, grad_B = grad_fn(A, B)
grad_A

loss_fn(A, B) = Array(963., dtype=float64)


(Array([[[28.],
         [38.]]], dtype=float64),
 Array([[[120., 108.],
         [ 92.,   0.]]], dtype=float64),
 Array([[[332., 332.,  92.,  28.],
         [ 24.,   0.,   0.,   0.]]], dtype=float64))

In [14]:
from tensordev.util.random_paths import random_trigonometric_polynomial_paths

n = 20
X = random_trigonometric_polynomial_paths(dim=d, batch=n, steps=20, key=jax.random.PRNGKey(0)).reshape(n, 1, 21, -1)

sig = core.tensor_path_signature(X, trunc=shuffle_core.trunc)

jnp.allclose(
    core.tensor_inner_product(sig, A) * core.tensor_inner_product(sig, B),
    core.tensor_inner_product(sig, res)
)

Array(True, dtype=bool)